# Lekcija 13 - Memorija agenta


## Postavljanje

Ovaj bilježnik pokazuje kako izgraditi agenta za rezervaciju putovanja s **stalnim memorijom** koristeći **Microsoft Agent Framework** (MAF).

Naučit ćete kako različite vrste memorije agenata — radna, kratkoročna i dugoročna — utječu na to kako agent zadržava i koristi informacije tijekom razgovora.

**Preduvjeti:**
- Microsoft Foundry projekt s implementiranim modelom za chat (npr. `gpt-5-mini`).
- Prijavljeni u Azure CLI — pokrenite `az login` u vašem terminalu.
- `AZURE_AI_PROJECT_ENDPOINT` — vaše krajnje odredište Microsoft Foundry projekta.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ime vašeg implementiranog modela.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Vrste memorije agenta

AI agenti mogu koristiti različite vrste memorije, od kojih svaka ima posebnu svrhu:

### Radna memorija
Sama nit razgovora — poruke razmijenjene u jednoj sesiji. Agent može pogledati ranije poruke u istoj niti radi održavanja koherentnosti. U MAF-u se ovo stvara putem **`agent.create_session()`**, što vraća `AgentSession`.

### Kratkoročna memorija
Informacije koje traju tijekom trajanja zadatka ili sesije, ali se ne pohranjuju trajno. Na primjer, agent može prikupljati činjenice tijekom višekratanog planiranja u razgovoru i koristiti ih za izradu konačnog itinerera.

### Dugoročna memorija
Preferencije i činjenice koje traju **preko sesija**. Povratni korisnik ne bi trebao ponovo navoditi svoje prehrambene restrikcije ili stil putovanja. Dugoročna memorija se obično podržava putem vanjskog spremišta — baze podataka, datoteke ili vektorskog indeksa — i dolazi do agenta putem alata.


## Radna memorija sa sesijama

Najjednostavniji oblik memorije je sesija razgovora. Kada proslijedite istu sesiju (koju ste stvorili pomoću `agent.create_session()`) uzastopnim pozivima `agent.run()`, agent vidi punu povijest tog razgovora i može se sjetiti ranijih detalja.

Napravimo agenta za putovanja i demonstrirajmo radnu memoriju.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agent se ispravno sjetio proračuna jer obje poruke dijele istu sesiju. Ovo je **radna memorija** — postoji samo tijekom trajanja sesije.

### Što se događa s novom niti?

Ako kreiramo **novu** sesiju, agent nema memoriju prethodnog razgovora:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Uzorak dugoročnog pamćenja

Za pamćenje korisničkih postavki **kroz sesije**, potrebna nam je trajna pohrana koja postoji izvan niti razgovora. Agent pristupa toj pohrani putem **alata** — funkcija koje može pozvati za spremanje i dohvaćanje informacija.

Ispod implementiramo jednostavnu memorijsku pohranu postavki (u produkciji biste to podržali bazom podataka ili vektorskim indeksom) i izlažemo je kao alate koje agent može koristiti.

### Arhitektura
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenarij 1 — Korisnik koji prvi put rezervira putovanje za obljetnicu

Sarah dolazi prvi put. Agent bi trebao pohraniti njezine preferencije putem alata i koristiti ih za preporuku hotela.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenarij 2 — Sarah se vraća tjednima kasnije

Sarah započinje **potpuno novu nit** (simulirajući novu sesiju). Radna memorija je prazna, ali pohrana dugoročnih preferencija i dalje ima njezine podatke. Agent bi ih trebao dohvatiti i koristiti za personalizirane preporuke.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Sažetak

U ovoj lekciji naučili ste tri tipa memorije agenta i kako ih implementirati s Microsoft Agent Frameworkom:

| Tip memorije | MAF mehanizam | Trajanje |
|---|---|---|
| **Radna** | `agent.create_session()` | Jedan razgovor |
| **Kratkoročna** | Akumulirani kontekst unutar thread-a | Jedan zadatak / sesija |
| **Dugoročna** | Vanjska pohrana dostupna preko `@tool` funkcija | Između sesija |

### Ključni zaključci
1. **`agent.create_session()`** pruža radnu memoriju — agent vidi cijelu povijest razgovora unutar sesije.
2. **Nove sesije gube kontekst** — bez dugoročne memorije agent ne može pamtiti prethodne razgovore.
3. **`@tool` funkcije** premošćuju jaz — omogućuju agentu spremanje i dohvaćanje informacija iz trajne pohrane.
4. **Personalizacija se poboljšava s vremenom** — što se više preferencija pohranjuje, bolje su preporuke agenta.

### Primjene u stvarnom svijetu
- **Korisnička služba**: Pamćenje povijesti i preferencija korisnika
- **Osobni pomoćnici**: Održavanje konteksta tijekom dana ili tjedana
- **Zdravstvo**: Praćenje informacija i preferencija pacijenata
- **E-trgovina**: Personalizirana kupovina na temelju povijesti

### Sljedeći koraci
- Zamijeniti in-memory dict bazom podataka ili spremištem vektora (npr. Azure AI Search)
- Dodati istekanje memorije za vremenski osjetljive informacije
- Izgraditi sustave s više agenata sa zajedničkom memorijom
- Istražiti Cognee bilježnicu za memoriju podržanu znanjem grafova


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
